In [17]:
import numpy as np
import pickle
from scipy.stats import hypergeom
import statsmodels.stats.multitest as smm
import sys

sys.path.append("/Users/sambe/Documents/Python_Projects/FDA/functions")
import fNIRSData

In [18]:
# Helper function to load data
def load_data(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

In [19]:
# User defined params
cptMethod = 'FF'
location = 'GM'
ageSpecific = 'N'
# Define the population size (number of channels)
populationSize = 34


In [20]:
if ageSpecific == 'N':
    tfceChannels = {
        'Total': [],
        'HbO': [],
        'HbR': [],
        'Both': []
    }
    for agePoint in [5,8,12]:

        tempChannels = fNIRSData.getROI(agePoint, location)

        for i in tempChannels['Total']:
            tfceChannels['Total'].append(i)
        for i in tempChannels['HbO']:
            tfceChannels['HbO'].append(i)
        for i in tempChannels['HbR']:
            tfceChannels['HbR'].append(i)
        for i in tempChannels['Both']:
            tfceChannels['Both'].append(i)

tfceChannels['Total'] = list(set(tfceChannels['Total']))
tfceChannels['HbO'] = list(set(tfceChannels['HbO']))
tfceChannels['HbR'] = list(set(tfceChannels['HbR']))
tfceChannels['Both'] = list(set(tfceChannels['Both']))

In [28]:
tfceChannels['Both']

[3, 4, 8, 9, 12, 13, 17, 18, 19]

In [21]:
agePoint = 5

cptFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/combinedChPt'+str(location)+str(agePoint)+'mo'+str(cptMethod)+'.pkl'

# Load CPt data
dfCPts = load_data(cptFilename)

# Define the lists
#TFCE
if ageSpecific == 'Y':
    tfceChannels = fNIRSData.getROI(agePoint, location)

tfceListTotal = tfceChannels["Total"]
tfceListHbo = tfceChannels["HbO"]
tfceListHbr = tfceChannels["HbR"]
tfceListBoth = tfceChannels["Both"]
    

# changepoint
cptListTotal = list(set(dfCPts['Channel'].to_list()))
dfCPtsHbo = dfCPts[dfCPts["Chromophore"] == 1]
cptListHbo = list(set(dfCPtsHbo['Channel'].to_list()))
dfCPtsHbr = dfCPts[dfCPts["Chromophore"] == 2]
cptListHbr = list(set(dfCPtsHbr['Channel'].to_list()))
# find all unique Chromophore values
uniqueChromophores = dfCPts['Chromophore'].unique()
# Group by 'Channel' and filter Channels where all Chromophore values are present
if (len(cptListHbo) == 0) or (len(cptListHbr) == 0):
    cptListBoth = []
else:
    dfCPtsBoth = dfCPts[
        dfCPts.groupby('Channel')['Chromophore']
        .transform(lambda x: len(x.unique()) == len(uniqueChromophores))
    ]
    cptListBoth = list(set(dfCPtsBoth['Channel'].to_list()))

# Hypergeometric distribution parameters
# N: population size
# M: number of successes in the population (size of TFCE list)
# n: number of draws (size of changepoints list)

# TOTAL CHANNELS
# Number of items in list1
M = len(tfceListTotal)
# Number of items in list2
n = len(cptListTotal)
# Number of overlaps
k = len(set(tfceListTotal) & set(cptListTotal))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue5total = hypergeom.sf(k-1, N, M, n)

print(f"Total channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue5total}")
print("")

# HbO CHANNELS
# Number of items in list1
M = len(tfceListHbo)
# Number of items in list2
n = len(cptListHbo)
# Number of overlaps
k = len(set(tfceListHbo) & set(cptListHbo))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue5hbo = hypergeom.sf(k-1, N, M, n)

print(f"HbO channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue5hbo}")
print("")

# HbR CHANNELS
# Number of items in list1
M = len(tfceListHbr)
# Number of items in list2
n = len(cptListHbr)
# Number of overlaps
k = len(set(tfceListHbr) & set(cptListHbr))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue5hbr = hypergeom.sf(k-1, N, M, n)

print(f"HbR channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue5hbr}")
print("")


# Both CHANNELS
# Number of items in list1
M = len(tfceListBoth)
# Number of items in list2
n = len(cptListBoth)
# Number of overlaps
k = len(set(tfceListBoth) & set(cptListBoth))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue5both = hypergeom.sf(k-1, N, M, n)

print(f"Both channels overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue5both}")

Total channel overlap for FF CPt at 5 months
Population size (N): 34
Number of successes in the population (M): 15
Number of draws (n): 20
Number of overlaps (k): 9
P-value: 0.5904515484193387

HbO channel overlap for FF CPt at 5 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 11
Number of overlaps (k): 5
P-value: 0.3146972419497448

HbR channel overlap for FF CPt at 5 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 16
Number of overlaps (k): 6
P-value: 0.5410762360755106

Both channels overlap for FF CPt at 5 months
Population size (N): 34
Number of successes in the population (M): 9
Number of draws (n): 7
Number of overlaps (k): 3
P-value: 0.25881698619381016


In [22]:
print(cptListTotal)
print(cptListHbo)
print(cptListHbr)
print(cptListBoth)

[2, 3, 6, 8, 9, 11, 12, 13, 14, 18, 19, 20, 23, 24, 25, 28, 29, 30, 33, 34]
[33, 6, 9, 11, 12, 18, 19, 23, 24, 28, 30]
[33, 2, 3, 34, 6, 8, 9, 11, 12, 13, 14, 18, 20, 24, 25, 29]
[33, 6, 9, 11, 12, 18, 24]


In [23]:
agePoint = 8

cptFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/combinedChPt'+str(location)+str(agePoint)+'mo'+str(cptMethod)+'.pkl'

# Load CPt data
dfCPts = load_data(cptFilename)

# Define the lists
#TFCE
if ageSpecific == 'Y':
    tfceChannels = fNIRSData.getROI(agePoint, location)

tfceListTotal = tfceChannels["Total"]
tfceListHbo = tfceChannels["HbO"]
tfceListHbr = tfceChannels["HbR"]
tfceListBoth = tfceChannels["Both"]

# changepoint
cptListTotal = list(set(dfCPts['Channel'].to_list()))
dfCPtsHbo = dfCPts[dfCPts["Chromophore"] == 1]
cptListHbo = list(set(dfCPtsHbo['Channel'].to_list()))
dfCPtsHbr = dfCPts[dfCPts["Chromophore"] == 2]
cptListHbr = list(set(dfCPtsHbr['Channel'].to_list()))
# find all unique Chromophore values
uniqueChromophores = dfCPts['Chromophore'].unique()
# Group by 'Channel' and filter Channels where all Chromophore values are present
if (len(cptListHbo) == 0) or (len(cptListHbr) == 0):
    cptListBoth = []
else:
    dfCPtsBoth = dfCPts[
        dfCPts.groupby('Channel')['Chromophore']
        .transform(lambda x: len(x.unique()) == len(uniqueChromophores))
    ]
    cptListBoth = list(set(dfCPtsBoth['Channel'].to_list()))

# Hypergeometric distribution parameters
# N: population size
# M: number of successes in the population (size of TFCE list)
# n: number of draws (size of changepoints list)

# TOTAL CHANNELS
# Number of items in list1
M = len(tfceListTotal)
# Number of items in list2
n = len(cptListTotal)
# Number of overlaps
k = len(set(tfceListTotal) & set(cptListTotal))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue8total = hypergeom.sf(k-1, N, M, n)

print(f"Total channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue8total}")
print("")

# HbO CHANNELS
# Number of items in list1
M = len(tfceListHbo)
# Number of items in list2
n = len(cptListHbo)
# Number of overlaps
k = len(set(tfceListHbo) & set(cptListHbo))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue8hbo = hypergeom.sf(k-1, N, M, n)

print(f"HbO channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue8hbo}")
print("")

# HbO CHANNELS
# Number of items in list1
M = len(tfceListHbr)
# Number of items in list2
n = len(cptListHbr)
# Number of overlaps
k = len(set(tfceListHbr) & set(cptListHbr))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue8hbr = hypergeom.sf(k-1, N, M, n)

print(f"HbR channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue8hbr}")
print("")


# Both CHANNELS
# Number of items in list1
M = len(tfceListBoth)
# Number of items in list2
n = len(cptListBoth)
# Number of overlaps
k = len(set(tfceListBoth) & set(cptListBoth))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue8both = hypergeom.sf(k-1, N, M, n)

print(f"Both channels overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue8both}")

Total channel overlap for FF CPt at 8 months
Population size (N): 34
Number of successes in the population (M): 15
Number of draws (n): 21
Number of overlaps (k): 9
P-value: 0.7069215392303848

HbO channel overlap for FF CPt at 8 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 14
Number of overlaps (k): 5
P-value: 0.6233754090696587

HbR channel overlap for FF CPt at 8 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 11
Number of overlaps (k): 5
P-value: 0.3146972419497448

Both channels overlap for FF CPt at 8 months
Population size (N): 34
Number of successes in the population (M): 9
Number of draws (n): 4
Number of overlaps (k): 3
P-value: 0.04799896498188718


In [24]:
print(cptListTotal)
print(cptListHbo)
print(cptListHbr)
print(cptListBoth)

[1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 14, 15, 17, 18, 20, 24, 25, 26, 27, 29, 30]
[1, 2, 3, 4, 8, 12, 14, 15, 18, 24, 26, 27, 29, 30]
[3, 6, 8, 9, 10, 11, 17, 18, 20, 25, 30]
[8, 18, 3, 30]


In [25]:
agePoint = 12

cptFilename = '/Users/sambe/Documents/Python_Projects/FDA/data/grpChangePointDFs/combinedChPt'+str(location)+str(agePoint)+'mo'+str(cptMethod)+'.pkl'

# Load CPt data
dfCPts = load_data(cptFilename)

# Define the lists
#TFCE
if ageSpecific == 'Y':
    tfceChannels = fNIRSData.getROI(agePoint, location)

tfceListTotal = tfceChannels["Total"]
tfceListHbo = tfceChannels["HbO"]
tfceListHbr = tfceChannels["HbR"]
tfceListBoth = tfceChannels["Both"]

# changepoint
cptListTotal = list(set(dfCPts['Channel'].to_list()))
dfCPtsHbo = dfCPts[dfCPts["Chromophore"] == 1]
cptListHbo = list(set(dfCPtsHbo['Channel'].to_list()))
dfCPtsHbr = dfCPts[dfCPts["Chromophore"] == 2]
cptListHbr = list(set(dfCPtsHbr['Channel'].to_list()))
# find all unique Chromophore values
uniqueChromophores = dfCPts['Chromophore'].unique()
# Group by 'Channel' and filter Channels where all Chromophore values are present
if (len(cptListHbo) == 0) or (len(cptListHbr) == 0):
    cptListBoth = []
else:
    dfCPtsBoth = dfCPts[
        dfCPts.groupby('Channel')['Chromophore']
        .transform(lambda x: len(x.unique()) == len(uniqueChromophores))
    ]
    cptListBoth = list(set(dfCPtsBoth['Channel'].to_list()))

# Hypergeometric distribution parameters
# N: population size
# M: number of successes in the population (size of TFCE list)
# n: number of draws (size of changepoints list)

# TOTAL CHANNELS
# Number of items in list1
M = len(tfceListTotal)
# Number of items in list2
n = len(cptListTotal)
# Number of overlaps
k = len(set(tfceListTotal) & set(cptListTotal))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue12total = hypergeom.sf(k-1, N, M, n)

print(f"Total channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue12total}")
print("")

# HbO CHANNELS
# Number of items in list1
M = len(tfceListHbo)
# Number of items in list2
n = len(cptListHbo)
# Number of overlaps
k = len(set(tfceListHbo) & set(cptListHbo))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue12hbo = hypergeom.sf(k-1, N, M, n)

print(f"HbO channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue12hbo}")
print("")

# HbO CHANNELS
# Number of items in list1
M = len(tfceListHbr)
# Number of items in list2
n = len(cptListHbr)
# Number of overlaps
k = len(set(tfceListHbr) & set(cptListHbr))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue12hbr = hypergeom.sf(k-1, N, M, n)

print(f"HbR channel overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue12hbr}")
print("")

# Both CHANNELS
# Number of items in list1
M = len(tfceListBoth)
# Number of items in list2
n = len(cptListBoth)
# Number of overlaps
k = len(set(tfceListBoth) & set(cptListBoth))
# number of channels
N = populationSize

# Calculate the p-value (probability of k or more overlaps by chance)
# Use survival function (sf) for P(X >= k)
pValue12both = hypergeom.sf(k-1, N, M, n)

print(f"Both channels overlap for {cptMethod} CPt at {agePoint} months")
print(f"Population size (N): {N}")
print(f"Number of successes in the population (M): {M}")
print(f"Number of draws (n): {n}")
print(f"Number of overlaps (k): {k}")
print(f"P-value: {pValue12both}")

Total channel overlap for FF CPt at 12 months
Population size (N): 34
Number of successes in the population (M): 15
Number of draws (n): 23
Number of overlaps (k): 11
P-value: 0.3996199480904709

HbO channel overlap for FF CPt at 12 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 17
Number of overlaps (k): 7
P-value: 0.36034079734326374

HbR channel overlap for FF CPt at 12 months
Population size (N): 34
Number of successes in the population (M): 12
Number of draws (n): 15
Number of overlaps (k): 7
P-value: 0.19177024391030292

Both channels overlap for FF CPt at 12 months
Population size (N): 34
Number of successes in the population (M): 9
Number of draws (n): 9
Number of overlaps (k): 4
P-value: 0.1619123477233796


In [26]:
print(cptListTotal)
print(cptListHbo)
print(cptListHbr)
print(cptListBoth)

[1, 2, 3, 5, 7, 8, 9, 12, 13, 17, 18, 19, 20, 21, 23, 27, 28, 29, 30, 31, 32, 33, 34]
[1, 2, 3, 34, 5, 8, 12, 17, 18, 19, 20, 21, 27, 28, 29, 30, 31]
[32, 1, 2, 3, 33, 5, 7, 9, 13, 17, 18, 19, 21, 23, 27]
[1, 2, 3, 5, 17, 18, 19, 21, 27]


Multiple comparison correction

In [27]:
print(f"{cptMethod} signficance")

#TOTAL CHANNELS
p_values = np.asarray([pValue5total, pValue8total, pValue12total]) #just for poster (FPCA only!)
# Bonferroni correction
bonferroni_corrected = [p * len(p_values) for p in p_values]
bonferroni_corrected = np.minimum(bonferroni_corrected, 1)  # Ensuring p-values don't exceed 1

# Benjamini-Hochberg correction
_, bh_corrected, _, _ = smm.multipletests(p_values, alpha=0.05, method='fdr_bh')

np.set_printoptions(suppress=True)

print("TOTAL CHANNELS p-Values")
print("BH:", bh_corrected)
print("")


#HbO CHANNELS
p_values = np.asarray([pValue5hbo, pValue8hbo, pValue12hbo]) #just for poster (FPCA only!)
# Bonferroni correction
bonferroni_corrected = [p * len(p_values) for p in p_values]
bonferroni_corrected = np.minimum(bonferroni_corrected, 1)  # Ensuring p-values don't exceed 1

# Benjamini-Hochberg correction
_, bh_corrected, _, _ = smm.multipletests(p_values, alpha=0.05, method='fdr_bh')

np.set_printoptions(suppress=True)

print("HbO CHANNELS p-Values")
print("BH:", bh_corrected)
print("")



#HbR CHANNELS
p_values = np.asarray([pValue5hbr, pValue8hbr, pValue12hbr]) #just for poster (FPCA only!)
# Bonferroni correction
bonferroni_corrected = [p * len(p_values) for p in p_values]
bonferroni_corrected = np.minimum(bonferroni_corrected, 1)  # Ensuring p-values don't exceed 1

# Benjamini-Hochberg correction
_, bh_corrected, _, _ = smm.multipletests(p_values, alpha=0.05, method='fdr_bh')

np.set_printoptions(suppress=True)

print("HbR CHANNELS p-Values")
print("BH:", bh_corrected)
print("")

#both CHANNELS
p_values = np.asarray([pValue5both, pValue8both, pValue12both]) #just for poster (FPCA only!)
# Bonferroni correction
bonferroni_corrected = [p * len(p_values) for p in p_values]
bonferroni_corrected = np.minimum(bonferroni_corrected, 1)  # Ensuring p-values don't exceed 1

# Benjamini-Hochberg correction
_, bh_corrected, _, _ = smm.multipletests(p_values, alpha=0.05, method='fdr_bh')

np.set_printoptions(suppress=True)

print("Both CHANNELS p-Values")
print("BH:", bh_corrected)

FF signficance
TOTAL CHANNELS p-Values
BH: [0.70692154 0.70692154 0.70692154]

HbO CHANNELS p-Values
BH: [0.5405112  0.62337541 0.5405112 ]

HbR CHANNELS p-Values
BH: [0.54107624 0.47204586 0.47204586]

Both CHANNELS p-Values
BH: [0.25881699 0.14399689 0.24286852]
